# Phase 4: Inference Testing & Validation — Synchronized

**Objective**: Evaluate the final model on unseen test data using the synchronized feature set.

### Key Metrics:
1. **Confusion Matrix**: Visualizing misclassifications.
2. **ROC/AUC Curves**: Assessing class separability.
3. **Inference Latency**: Measuring execution time.

### Notes:
- Model: `best_FaultMLP.pth` (trained by `scripts/model_training.py`)
- Scaler: loaded from `scaler.joblib` — **do NOT refit** to avoid train/test leakage
- Split: 80/20 stratified (identical to `model_training.py`)
- Labels: B=0, IR=1, Normal=2, OR=3 (alphabetical)

In [ ]:
import os, time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH     = "../data/models/best_FaultMLP.pth"   # matches model_training.py output
SCALER_PATH    = "../data/models/scaler.joblib"        # load — never refit
PROCESSED_PATH = "../data/processed/cwru_features.parquet"
CLASS_NAMES    = {0: 'B', 1: 'IR', 2: 'Normal', 3: 'OR'}

print(f"Inference Testing active on: {DEVICE}")

## 1. Load Data, Scaler, and Model

In [ ]:
df = pd.read_parquet(PROCESSED_PATH)

# Drop metadata columns (same logic as model_training.py)
meta_cols = ['fault_type', 'fault_class', 'fault_diameter', 'load', 'rpm', 'sampling_rate_hz']
df = df.drop(columns=[c for c in meta_cols if c in df.columns])

# Same 80/20 stratified split as model_training.py
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['label']), df['label'],
    test_size=0.2, stratify=df['label'], random_state=42
)

# Load production scaler — DO NOT refit
scaler = joblib.load(SCALER_PATH)
X_test_scaled = scaler.transform(X_test)

# Rebuild FaultMLP architecture (must match model_training.py)
class FaultMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x): return self.net(x)

num_classes = len(df['label'].unique())
model = FaultMLP(X_test_scaled.shape[1], num_classes).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print(f"Model loaded | Classes: {num_classes} | Input dim: {X_test_scaled.shape[1]}")

## 2. Inference + Latency

In [ ]:
inputs = torch.FloatTensor(X_test_scaled).to(DEVICE)

t0 = time.perf_counter()
with torch.no_grad():
    outputs = model(inputs)
    probs   = torch.softmax(outputs, dim=1)
    _, preds = torch.max(outputs, 1)
t1 = time.perf_counter()

preds = preds.cpu().numpy()
probs = probs.cpu().numpy()
print(f"Inference time: {(t1-t0)*1000:.1f}ms | "
      f"Per sample: {(t1-t0)/len(preds)*1000:.3f}ms")

## 3. Confusion Matrix & Classification Report

In [ ]:
labels = sorted(df['label'].unique())
class_names = [CLASS_NAMES[l] for l in labels]

print(classification_report(y_test, preds, target_names=class_names))

cm = confusion_matrix(y_test, preds, labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Test Set (80/20 split)')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

## 4. ROC Curves (One-vs-Rest)

In [ ]:
y_bin = label_binarize(y_test, classes=labels)
plt.figure(figsize=(8, 6))
colors = ['#e6194b', '#3cb44b', '#4363d8', '#f58231']

for i, (cls, name) in enumerate(zip(labels, class_names)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — 4-Class (One-vs-Rest)')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()